# to_craft research

this notebook is a design/prototyping notebook for converting the project rocket dict format into a KSP1 `.craft` file.

goal for this notebook:

- show the conversion pipeline i would use
- demonstrate what data we can already generate from the rocket dict
- make the missing pieces explicit before attempting a real KSP integration


## current project representation

the current rocket dict already gives us:

- part order / tree structure
- project part ids like `pod_0`, `tank_0`, `eng_0`
- parent links
- project stage numbers

what it does **not** directly give us:

- KSP-style unique part ids in the `.craft` file
- explicit 3D placement (`pos`, `attPos0`, etc.)
- KSP staging fields like `istg` / `dstg`
- per-part `MODULE` and `RESOURCE` block serialization strategy


In [9]:
from pathlib import Path
import random

from src.config import load_parts_by_name

parts_by_name = load_parts_by_name()

sample_rocket = {
    'parts': [
        {'id': 'pod_0', 'type': 'probeCoreOcto_v2', 'parent': None, 'attach_node': None},
        {'id': 'tank_0', 'type': 'adapterSize2-Size1', 'parent': 'pod_0', 'attach_node': 'bottom'},
        {'id': 'eng_0', 'type': 'microEngine_v2', 'parent': 'tank_0', 'attach_node': 'bottom'},
        {'id': 'decoupler_0', 'type': 'Decoupler_0', 'parent': 'eng_0', 'attach_node': 'bottom'},
        {'id': 'tank_1', 'type': 'Size3SmallTank', 'parent': 'decoupler_0', 'attach_node': 'bottom'},
        {'id': 'eng_1', 'type': 'Size2LFB_v2', 'parent': 'tank_1', 'attach_node': 'bottom'},
    ],
    'stages': {
        'eng_0': 0,
        'decoupler_0': 1,
        'eng_1': 1,
    }
}

sample_rocket

{'parts': [{'id': 'pod_0',
   'type': 'probeCoreOcto_v2',
   'parent': None,
   'attach_node': None},
  {'id': 'tank_0',
   'type': 'adapterSize2-Size1',
   'parent': 'pod_0',
   'attach_node': 'bottom'},
  {'id': 'eng_0',
   'type': 'microEngine_v2',
   'parent': 'tank_0',
   'attach_node': 'bottom'},
  {'id': 'decoupler_0',
   'type': 'Decoupler_0',
   'parent': 'eng_0',
   'attach_node': 'bottom'},
  {'id': 'tank_1',
   'type': 'Size3SmallTank',
   'parent': 'decoupler_0',
   'attach_node': 'bottom'},
  {'id': 'eng_1',
   'type': 'Size2LFB_v2',
   'parent': 'tank_1',
   'attach_node': 'bottom'}],
 'stages': {'eng_0': 0, 'decoupler_0': 1, 'eng_1': 1}}

## conversion plan

for a first-pass linear-stack `to_craft()` conversion, the pipeline would be:

1. assign each project part a KSP `.craft` part id like `probeCoreOcto.v2_123456789`
2. derive child links from the parent tree
3. place the root at a fixed editor position, then use part node geometry to stack children
4. serialize top-level vessel metadata
5. serialize part identity / placement / connectivity fields
6. translate project stage numbers into KSP staging fields
7. fill in or template required `MODULE` / `RESOURCE` blocks

steps 1 through 5 are straightforward enough to prototype now. steps 6 and 7 are still the risky ones.


In [10]:
def make_ksp_part_ids(rocket_dict, seed=12345):
    rng = random.Random(seed)
    id_map = {}
    for part in rocket_dict['parts']:
        suffix = rng.randint(1_000_000_000, 4_294_967_295)
        id_map[part['id']] = f"{part['type']}_{suffix}"
    return id_map


def build_children_lookup(rocket_dict):
    children = {part['id']: [] for part in rocket_dict['parts']}
    for part in rocket_dict['parts']:
        parent = part['parent']
        if parent is not None:
            children[parent].append(part['id'])
    return children


def linear_stack_positions(rocket_dict, parts_by_name, root_pos=(0.0, 15.0, 0.0)):
    """Place a linear rocket stack by aligning parent/child attach nodes.

    For a parent-child stack connection, we want:
        child_world_pos + child_top_node == parent_world_pos + parent_attach_node

    So:
        child_world_pos = parent_world_node - child_local_top_node

    This is enough for a first-pass stack-only prototype because the parts library
    already contains node positions for all scraped parts.
    """
    positions = {}
    attach_offsets = {}

    parts_by_id = {part['id']: part for part in rocket_dict['parts']}
    root = next(part['id'] for part in rocket_dict['parts'] if part['parent'] is None)
    positions[root] = root_pos
    attach_offsets[root] = (0.0, 0.0, 0.0)

    current = root
    while True:
        children = [part['id'] for part in rocket_dict['parts'] if part['parent'] == current]
        if not children:
            break

        child = children[0]
        parent_part = parts_by_id[current]
        child_part = parts_by_id[child]

        parent_attach_name = child_part.get('attach_node', 'bottom') or 'bottom'
        child_attach_name = 'top'

        parent_attach = parts_by_name[parent_part['type']]['nodes'][parent_attach_name]['pos']
        child_attach = parts_by_name[child_part['type']]['nodes'][child_attach_name]['pos']

        px, py, pz = positions[current]
        parent_node_world = (
            px + parent_attach[0],
            py + parent_attach[1],
            pz + parent_attach[2],
        )

        child_world = (
            parent_node_world[0] - child_attach[0],
            parent_node_world[1] - child_attach[1],
            parent_node_world[2] - child_attach[2],
        )

        positions[child] = child_world
        attach_offsets[child] = tuple(parent_attach)
        current = child

    return positions, attach_offsets


ksp_id_map = make_ksp_part_ids(sample_rocket)
children_lookup = build_children_lookup(sample_rocket)
positions, attach_offsets = linear_stack_positions(sample_rocket, parts_by_name)

print('ksp part ids:')
for k, v in ksp_id_map.items():
    print(' ', k, '->', v)

print('\nchildren lookup:')
for k, v in children_lookup.items():
    print(' ', k, '->', v)

print('\nnode-based positions:')
for k, v in positions.items():
    print(' ', k, '->', v)

print('\nparent attach offsets used:')
for k, v in attach_offsets.items():
    print(' ', k, '->', v)


ksp part ids:
  pod_0 -> probeCoreOcto_v2_2789368711
  tank_0 -> adapterSize2-Size1_4146859322
  eng_0 -> microEngine_v2_1043676229
  decoupler_0 -> Decoupler_0_2282648386
  tank_1 -> Size3SmallTank_2582316135
  eng_1 -> Size2LFB_v2_1831769172

children lookup:
  pod_0 -> ['tank_0']
  tank_0 -> ['eng_0']
  eng_0 -> ['decoupler_0']
  decoupler_0 -> ['tank_1']
  tank_1 -> ['eng_1']
  eng_1 -> []

node-based positions:
  pod_0 -> (0.0, 15.0, 0.0)
  tank_0 -> (0.0, 13.5629182, 0.0)
  eng_0 -> (0.0, 12.3129182, 0.0)
  decoupler_0 -> (0.0, 11.9987197, 0.0)
  tank_1 -> (0.0, 11.006219699999999, 0.0)
  eng_1 -> (0.0, 5.683219699999999, 0.0)

parent attach offsets used:
  pod_0 -> (0.0, 0.0, 0.0)
  tank_0 -> (0.0, -0.1870818, 0.0)
  eng_0 -> (0.0, -1.25, 0.0)
  decoupler_0 -> (0.0, -0.2816985, 0.0)
  tank_1 -> (0.0, -0.0325, 0.0)
  eng_1 -> (0.0, -0.967, 0.0)


In [11]:
def render_prototype_header(ship_name='prototype craft'):
    return '\n'.join([
        f'ship = {ship_name}',
        'version = 1.6.0',
        'description = generated prototype',
        'type = VAB',
        'size = 1,1,1',
        'steamPublishedFileId = 0',
        'persistentId = 123456789',
        'rot = 0,0,0,1',
        'missionFlag = Squad/Flags/default',
        'vesselType = Debris',
    ])


def render_prototype_part_blocks(rocket_dict, parts_by_name, ksp_id_map, positions, attach_offsets):
    """Render the fields we can now derive from the rocket dict plus node geometry.

    This still is not guaranteed to be a valid full `.craft` serializer because
    staging translation and `MODULE` / `RESOURCE` templating are unresolved.
    But the stack placement itself is now based on real node geometry instead of a fake step size.
    """
    blocks = []
    children_lookup = build_children_lookup(rocket_dict)
    for part in rocket_dict['parts']:
        part_id = part['id']
        ksp_id = ksp_id_map[part_id]
        x, y, z = positions[part_id]
        child_links = [ksp_id_map[c] for c in children_lookup[part_id]]
        stage = rocket_dict['stages'].get(part_id)

        lines = [
            'PART',
            '{',
            f'\tpart = {ksp_id}',
            '\tpartName = Part',
            '\tpersistentId = 123456789',
            f'\tpos = {x},{y},{z}',
            '\tattPos = 0,0,0',
            '\t; attPos0 here is a prototype parent-node world position, not yet KSP-validated',
            '\trot = 0,0,0,1',
            '\tattRot = 0,0,0,1',
            '\tattRot0 = 0,0,0,1',
            '\tmir = 1,1,1',
            '\tsymMethod = Radial',
            '\tautostrutMode = Off',
            '\trigidAttachment = False',
            f'\t; TODO project stage = {stage}',
            '\t; TODO derive istg / dstg / sidx / sqor / sepI',
        ]

        for child_id in child_links:
            lines.append(f'\tlink = {child_id}')

        if part['parent'] is not None:
            parent_ksp_id = ksp_id_map[part['parent']]
            node = part.get('attach_node', 'bottom') or 'bottom'
            ax, ay, az = attach_offsets[part_id]
            lines.append(f'\tattPos0 = {x + parts_by_name[part["type"]]["nodes"]["top"]["pos"][0]},{y + parts_by_name[part["type"]]["nodes"]["top"]["pos"][1]},{z + parts_by_name[part["type"]]["nodes"]["top"]["pos"][2]}')
            lines.append(f'\t; attN offset is currently derived from parent node geometry')
            lines.append(f'\tattN = {node},{parent_ksp_id}_0|{ay}|0')
        else:
            lines.append(f'\tattPos0 = {x},{y},{z}')

        lines.extend([
            '\tEVENTS',
            '\t{',
            '\t}',
            '\tACTIONS',
            '\t{',
            '\t}',
            '\tPARTDATA',
            '\t{',
            '\t}',
            '\t; TODO inject MODULE blocks from a safe template source',
            '\t; TODO inject RESOURCE blocks where needed',
            '}',
        ])
        blocks.append('\n'.join(lines))

    return '\n'.join(blocks)


prototype_craft_text = render_prototype_header() + '\n' + render_prototype_part_blocks(sample_rocket, parts_by_name, ksp_id_map, positions, attach_offsets)
print(prototype_craft_text[:3000])


ship = prototype craft
version = 1.6.0
description = generated prototype
type = VAB
size = 1,1,1
steamPublishedFileId = 0
persistentId = 123456789
rot = 0,0,0,1
missionFlag = Squad/Flags/default
vesselType = Debris
PART
{
	part = probeCoreOcto_v2_2789368711
	partName = Part
	persistentId = 123456789
	pos = 0.0,15.0,0.0
	attPos = 0,0,0
	; attPos0 here is a prototype parent-node world position, not yet KSP-validated
	rot = 0,0,0,1
	attRot = 0,0,0,1
	attRot0 = 0,0,0,1
	mir = 1,1,1
	symMethod = Radial
	autostrutMode = Off
	rigidAttachment = False
	; TODO project stage = None
	; TODO derive istg / dstg / sidx / sqor / sepI
	link = adapterSize2-Size1_4146859322
	attPos0 = 0.0,15.0,0.0
	EVENTS
	{
	}
	ACTIONS
	{
	}
	PARTDATA
	{
	}
	; TODO inject MODULE blocks from a safe template source
	; TODO inject RESOURCE blocks where needed
}
PART
{
	part = adapterSize2-Size1_4146859322
	partName = Part
	persistentId = 123456789
	pos = 0.0,13.5629182,0.0
	attPos = 0,0,0
	; attPos0 here is a prototype par

## what this prototype proves

we can already derive and serialize:

- a vessel-level header
- deterministic KSP-style part ids
- linear-stack part ordering
- parent/child `link` relationships
- node-based `attN` / parent-offset relationships
- node-based stack positions derived from the scraped parts library

that means the overall **shape** of `to_craft()` is not the blocker anymore.


## the missing pieces

these are the things we still need to figure out before `to_craft()` can produce a real loadable file:

1. exact `.craft` transform semantics
   - we now have node geometry from the parts library
   - but we still need to confirm that our derived `pos` / `attPos0` fields match what KSP expects in a real craft file

2. staging translation
   - our project uses stage `0` as the last-to-fire stage
   - KSP `.craft` files use `istg`, `dstg`, and related fields
   - we still need a reliable mapping from project stages to those KSP fields

3. module serialization strategy
   - every stock part has `MODULE { ... }` blocks in the `.craft` file
   - we do not yet know which fields can be defaulted safely and which must be preserved from a template

4. resource serialization strategy
   - some parts carry explicit `RESOURCE { ... }` blocks in the craft file
   - we need to decide whether to generate them from the parts library or copy them from known-good templates

5. validation method
   - the real proof is not text generation
   - the real proof is: can KSP load the file without errors?


## template-based approach

the scraped part data is rich enough for geometry and resources, but not rich enough to synthesize full `.craft` part blocks from scratch.

so the practical plan is:

- generate vessel-specific fields ourselves
- reuse real stock `.craft` part blocks as templates for the missing boilerplate

this still generalizes to GA rockets because the templates are per **part type**, not per whole vessel.


In [12]:
def extract_part_blocks(craft_text):
    """Return raw PART blocks from a craft file by brace counting."""
    lines = craft_text.splitlines()
    blocks = []
    i = 0
    while i < len(lines):
        if lines[i].strip() == 'PART':
            block_lines = [lines[i]]
            i += 1
            brace_depth = 0
            while i < len(lines):
                line = lines[i]
                block_lines.append(line)
                brace_depth += line.count('{')
                brace_depth -= line.count('}')
                i += 1
                if brace_depth == 0 and line.strip() == '}':
                    break
            blocks.append('\n'.join(block_lines))
        else:
            i += 1
    return blocks


def find_template_blocks_for_parts(part_names, search_root):
    """Search local craft files for PART blocks matching requested internal part names."""
    search_root = Path(search_root)
    wanted = {name: None for name in part_names}
    craft_files = sorted(search_root.rglob('*.craft'))

    for craft_file in craft_files:
        if all(wanted[name] is not None for name in wanted):
            break
        text = craft_file.read_text(errors='ignore')
        for block in extract_part_blocks(text):
            for name in wanted:
                if wanted[name] is None and f'part = {name}_' in block:
                    wanted[name] = {'file': craft_file, 'block': block}
    return wanted


sample_part_names = [part['type'] for part in sample_rocket['parts']]
templates = find_template_blocks_for_parts(
    sample_part_names,
    '/Users/moss/Library/Application Support/Steam/steamapps/common/Kerbal Space Program',
)

print('template coverage for sample rocket:')
for name, result in templates.items():
    if result is None:
        print(f'  {name}: MISSING')
    else:
        print(f'  {name}: found in {result["file"].name}')


template coverage for sample rocket:
  probeCoreOcto_v2: MISSING
  adapterSize2-Size1: MISSING
  microEngine_v2: MISSING
  Decoupler_0: MISSING
  Size3SmallTank: found in Dynawing.craft
  Size2LFB_v2: MISSING


In [13]:
for name, result in templates.items():
    if result is not None:
        print(f'=== TEMPLATE EXAMPLE: {name} ===')
        print(result['block'][:2000])
        print()
        break


=== TEMPLATE EXAMPLE: Size3SmallTank ===
PART
{
	part = Size3SmallTank_4294215638
	partName = Part
	persistentId = 1972509124
	pos = 0.188970566,22.849123,-0.64718467
	attPos = 0,0,0
	attPos0 = -4.18568256E-08,4.70699978,-4.88631031E-07
	rot = -1.11758709E-07,-3.27825546E-07,-0.0767018571,0.997054219
	attRot = 0,0,0,1
	attRot0 = -1.60727609E-09,0.70710659,-4.47417037E-09,0.707106948
	mir = 1,1,1
	symMethod = Radial
	autostrutMode = Off
	rigidAttachment = False
	istg = 1
	resPri = -1
	dstg = 2
	sidx = -1
	sqor = -1
	sepI = 1
	attm = 0
	modCost = 0
	modMass = 0
	modSize = 0,0,0
	link = Size3SmallTank_4294214788
	attN = top,Size3SmallTank_4294214788_0|0.96|0
	attN = bottom,Size3LargeTank_4294103130_0|-0.967|0
	EVENTS
	{
	}
	ACTIONS
	{
	}
	PARTDATA
	{
	}
	RESOURCE
	{
		name = LiquidFuel
		amount = 1620
		maxAmount = 1620
		flowState = True
		isTweakable = True
		hideFlow = False
		isVisible = True
		flowMode = Both
	}
	RESOURCE
	{
		name = Oxidizer
		amount = 1980
		maxAmount = 1980
		flow

In [14]:
open_questions = [
    'How do we map project stages onto KSP istg/dstg/sidx/sqor/sepI fields?',
    'Can we reuse per-part MODULE and RESOURCE templates from stock craft files?',
    'What is the minimum attach-node / position data needed for a linear stack to load?',
    'Do we need per-part templates by internal part name, or can we generate most fields generically?',
]

print('open questions before real to_craft():')
for i, q in enumerate(open_questions, start=1):
    print(f'{i}. {q}')


open questions before real to_craft():
1. How do we map project stages onto KSP istg/dstg/sidx/sqor/sepI fields?
2. Can we reuse per-part MODULE and RESOURCE templates from stock craft files?
3. What is the minimum attach-node / position data needed for a linear stack to load?
4. Do we need per-part templates by internal part name, or can we generate most fields generically?


## staging translation for a linear stack

stock VAB craft suggest that our current project stages do not map one-to-one onto KSP activation stages.

for a simple stacked rocket, KSP separates:

- decoupler activation
- engine activation below that decoupler

so the working first-pass rule for our current linear-stack rockets is:

- project stage `0`:
  - engine `istg = 0`, `dstg = 0`
  - passive parts `istg = -1`, `dstg = 0`
- project stage `k > 0`:
  - decoupler: `istg = 2k - 1`, `dstg = 2k - 1`
  - engine: `istg = 2k`, `dstg = 2k`
  - passive parts in that stage: `istg = 2k - 1`, `dstg = 2k`


In [ ]:
def project_part_stage_context(part, rocket_dict):
    """Classify a part into linear-stack project-stage context."""
    part_id = part['id']
    project_stage = rocket_dict['stages'].get(part_id)

    if project_stage is not None:
        if part_id.startswith('decoupler_'):
            return {'project_stage': project_stage, 'role': 'decoupler'}
        if part_id.startswith('eng_'):
            return {'project_stage': project_stage, 'role': 'engine'}

    current = part['id']
    while True:
        children = [p['id'] for p in rocket_dict['parts'] if p['parent'] == current]
        if not children:
            break
        child_id = children[0]
        if child_id in rocket_dict['stages']:
            return {'project_stage': rocket_dict['stages'][child_id], 'role': 'passive'}
        current = child_id

    return {'project_stage': 0, 'role': 'passive'}


def translate_staging_linear(part, rocket_dict):
    """Translate project stages into KSP istg/dstg/etc for a linear stack."""
    ctx = project_part_stage_context(part, rocket_dict)
    stage = ctx['project_stage']
    role = ctx['role']

    if stage == 0:
        if role == 'engine':
            return {'istg': 0, 'dstg': 0, 'sidx': 0, 'sqor': 0, 'sepI': -1}
        return {'istg': -1, 'dstg': 0, 'sidx': -1, 'sqor': -1, 'sepI': -1}

    if role == 'decoupler':
        ksp_stage = 2 * stage - 1
        return {'istg': ksp_stage, 'dstg': ksp_stage, 'sidx': 0, 'sqor': ksp_stage, 'sepI': ksp_stage}

    if role == 'engine':
        return {'istg': 2 * stage, 'dstg': 2 * stage, 'sidx': 0, 'sqor': 2 * stage, 'sepI': 2 * stage - 1}

    return {'istg': 2 * stage - 1, 'dstg': 2 * stage, 'sidx': -1, 'sqor': -1, 'sepI': 2 * stage - 1}


print('sample rocket staging translation:')
for part in sample_rocket['parts']:
    print(part['id'], part['type'], translate_staging_linear(part, sample_rocket))


## final-shape prototype function

these cells move from research helpers toward the actual shape of the future `to_craft()` implementation:

- parse template blocks into structure
- override the fields we control
- synthesize fallback blocks when templates are missing
- render the final `.craft` text
- return `(craft_text, metadata)`


In [15]:
def parse_part_template_block(block_text):
    """Parse one raw PART block into top-level fields plus preserved nested blocks."""
    lines = block_text.splitlines()
    if not lines or lines[0].strip() != 'PART':
        raise ValueError('expected PART block')

    fields = []
    nested_blocks = []
    i = 2
    while i < len(lines) - 1:
        line = lines[i]
        stripped = line.strip()

        if not stripped:
            i += 1
            continue

        if stripped.endswith('{'):
            block_lines = [line]
            brace_depth = line.count('{') - line.count('}')
            i += 1
            while i < len(lines):
                block_lines.append(lines[i])
                brace_depth += lines[i].count('{') - lines[i].count('}')
                i += 1
                if brace_depth == 0:
                    break
            nested_blocks.append('\n'.join(block_lines))
            continue

        if '=' in stripped:
            key, value = stripped.split('=', 1)
            fields.append((key.strip(), value.strip()))

        i += 1

    return {'fields': fields, 'nested_blocks': nested_blocks}


def render_part_template_struct(part_struct):
    lines = ['PART', '{']
    for key, value in part_struct['fields']:
        lines.append(f'\t{key} = {value}')
    for block in part_struct['nested_blocks']:
        for line in block.splitlines():
            lines.append(line)
    lines.append('}')
    return '\n'.join(lines)


In [16]:
def make_generic_part_struct(part, rocket_dict, parts_by_name, ksp_id_map, positions, attach_offsets):
    """Build a generic PART structure when no stock template is available."""
    x, y, z = positions[part['id']]
    stage = rocket_dict['stages'].get(part['id'])
    fields = [
        ('part', ksp_id_map[part['id']]),
        ('partName', 'Part'),
        ('persistentId', '123456789'),
        ('pos', f'{x},{y},{z}'),
        ('attPos', '0,0,0'),
        ('attPos0', f'{x},{y},{z}'),
        ('rot', '0,0,0,1'),
        ('attRot', '0,0,0,1'),
        ('attRot0', '0,0,0,1'),
        ('mir', '1,1,1'),
        ('symMethod', 'Radial'),
        ('autostrutMode', 'Off'),
        ('rigidAttachment', 'False'),
        ('resPri', '0'),
        ('modCost', '0'),
        ('modMass', '0'),
        ('modSize', '0,0,0'),
    ]

    staging = translate_staging_linear(part, rocket_dict)
    fields.extend([
        ('istg', str(staging['istg'])),
        ('dstg', str(staging['dstg'])),
        ('sidx', str(staging['sidx'])),
        ('sqor', str(staging['sqor'])),
        ('sepI', str(staging['sepI'])),
    ])

    children = build_children_lookup(rocket_dict)[part['id']]
    for child_id in children:
        fields.append(('link', ksp_id_map[child_id]))

    if part['parent'] is not None:
        parent_ksp_id = ksp_id_map[part['parent']]
        node = part.get('attach_node', 'bottom') or 'bottom'
        ax, ay, az = attach_offsets[part['id']]
        fields.append(('attN', f'{node},{parent_ksp_id}_0|{ay}|0'))

    nested_blocks = [
        '\tEVENTS\n\t{\n\t}',
        '\tACTIONS\n\t{\n\t}',
        '\tPARTDATA\n\t{\n\t}',
        '\t; FALLBACK generic block: MODULE / RESOURCE templating still unresolved for this part type',
    ]

    return {'fields': fields, 'nested_blocks': nested_blocks}


def apply_common_overrides(part_struct, part, rocket_dict, ksp_id_map, positions, attach_offsets):
    """Override the vessel-specific fields in a parsed template structure."""
    x, y, z = positions[part['id']]
    stage = rocket_dict['stages'].get(part['id'])
    children = build_children_lookup(rocket_dict)[part['id']]

    field_map = []
    skip_keys = {'link', 'attN', 'part', 'persistentId', 'pos', 'attPos0', 'istg', 'dstg', 'sidx', 'sqor', 'sepI'}
    for key, value in part_struct['fields']:
        if key not in skip_keys:
            field_map.append((key, value))

    field_map.insert(0, ('part', ksp_id_map[part['id']]))
    field_map.insert(1, ('persistentId', '123456789'))
    field_map.insert(2, ('pos', f'{x},{y},{z}'))
    field_map.insert(3, ('attPos0', f'{x},{y},{z}'))

    staging = translate_staging_linear(part, rocket_dict)
    field_map.extend([
        ('istg', str(staging['istg'])),
        ('dstg', str(staging['dstg'])),
        ('sidx', str(staging['sidx'])),
        ('sqor', str(staging['sqor'])),
        ('sepI', str(staging['sepI'])),
    ])

    for child_id in children:
        field_map.append(('link', ksp_id_map[child_id]))

    if part['parent'] is not None:
        parent_ksp_id = ksp_id_map[part['parent']]
        node = part.get('attach_node', 'bottom') or 'bottom'
        ax, ay, az = attach_offsets[part['id']]
        field_map.append(('attN', f'{node},{parent_ksp_id}_0|{ay}|0'))

    return {'fields': field_map, 'nested_blocks': part_struct['nested_blocks']}


In [17]:
def to_craft(rocket_dict, parts_by_name, ship_name='prototype craft', search_root='/Users/moss/Library/Application Support/Steam/steamapps/common/Kerbal Space Program'):
    """Notebook prototype of the eventual `to_craft()` function.

    Returns
    -------
    tuple
        (craft_text, metadata)
    """
    ksp_id_map = make_ksp_part_ids(rocket_dict)
    positions, attach_offsets = linear_stack_positions(rocket_dict, parts_by_name)
    templates = find_template_blocks_for_parts([part['type'] for part in rocket_dict['parts']], search_root)

    metadata = {
        'template_parts': [],
        'fallback_parts': [],
        'warnings': [],
    }

    part_blocks = []
    for part in rocket_dict['parts']:
        template_info = templates.get(part['type'])
        if template_info is not None:
            parsed = parse_part_template_block(template_info['block'])
            updated = apply_common_overrides(parsed, part, rocket_dict, ksp_id_map, positions, attach_offsets)
            metadata['template_parts'].append(part['type'])
        else:
            updated = make_generic_part_struct(part, rocket_dict, parts_by_name, ksp_id_map, positions, attach_offsets)
            metadata['fallback_parts'].append(part['type'])
            metadata['warnings'].append(f'used generic fallback block for {part["type"]}')

        part_blocks.append(render_part_template_struct(updated))

    craft_text = render_prototype_header(ship_name=ship_name) + '\n' + '\n'.join(part_blocks)
    return craft_text, metadata


craft_text, craft_meta = to_craft(sample_rocket, parts_by_name)
print(craft_text[:3500])
print('\nmetadata:')
print(craft_meta)


ship = prototype craft
version = 1.6.0
description = generated prototype
type = VAB
size = 1,1,1
steamPublishedFileId = 0
persistentId = 123456789
rot = 0,0,0,1
missionFlag = Squad/Flags/default
vesselType = Debris
PART
{
	part = probeCoreOcto_v2_2789368711
	partName = Part
	persistentId = 123456789
	pos = 0.0,15.0,0.0
	attPos = 0,0,0
	attPos0 = 0.0,15.0,0.0
	rot = 0,0,0,1
	attRot = 0,0,0,1
	attRot0 = 0,0,0,1
	mir = 1,1,1
	symMethod = Radial
	autostrutMode = Off
	rigidAttachment = False
	resPri = 0
	modCost = 0
	modMass = 0
	modSize = 0,0,0
	istg = -1
	dstg = 0
	sidx = -1
	sqor = -1
	sepI = -1
	link = adapterSize2-Size1_4146859322
	EVENTS
	{
	}
	ACTIONS
	{
	}
	PARTDATA
	{
	}
	; FALLBACK generic block: MODULE / RESOURCE templating still unresolved for this part type
}
PART
{
	part = adapterSize2-Size1_4146859322
	partName = Part
	persistentId = 123456789
	pos = 0.0,13.5629182,0.0
	attPos = 0,0,0
	attPos0 = 0.0,13.5629182,0.0
	rot = 0,0,0,1
	attRot = 0,0,0,1
	attRot0 = 0,0,0,1
	mir = 1,1